# DX 704 Week 9 Project

This week's project will build an email spam classifier based on the Enron email data set.
You will perform your own feature extraction, and use naive Bayes to estimate the probability that a particular email is spam or not.
Finally, you will review the tradeoffs from different thresholds for automatically sending emails to the junk folder.

The full project description and a template notebook are available on GitHub: [Project 9 Materials](https://github.com/bu-cds-dx704/dx704-project-09).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Download Data Set

We will be using the Enron spam data set as prepared in this GitHub repository.

https://github.com/MWiechmann/enron_spam_data

You may need to download this differently depending on your environment.

In [1]:
# import urllib.request

# url = "https://github.com/MWiechmann/enron_spam_data/raw/refs/heads/master/enron_spam_data.zip"
# urllib.request.urlretrieve(url, "enron_spam_data.zip")

In [2]:
import pandas as pd
import numpy as np
import json
from collections import defaultdict
import re


In [3]:
# pandas can read the zip file directly
enron_spam_data = pd.read_csv("enron_spam_data.zip")
enron_spam_data

,Message ID,Subject,Message,Spam/Ham,Date
0,0,christmas tree farm pictures,NaN,ham,1999-12-10
1,1,"vastar resources , inc .","gary , production from the high island larger ...",ham,1999-12-13
2,2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham,1999-12-14
3,3,re : issue,fyi - see note below - already done .\nstella\...,ham,1999-12-14
4,4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham,1999-12-14
...,...,...,...,...,...
33711,33711,= ? iso - 8859 - 1 ? q ? good _ news _ c = eda...,"hello , welcome to gigapharm onlinne shop .\np...",spam,2005-07-29
33712,33712,all prescript medicines are on special . to be...,i got it earlier than expected and it was wrap...,spam,2005-07-29
33713,33713,the next generation online pharmacy .,are you ready to rock on ? let the man in you ...,spam,2005-07-30
33714,33714,bloow in 5 - 10 times the time,learn how to last 5 - 10 times longer in\nbed ...,spam,2005-07-30


In [4]:
(enron_spam_data["Spam/Ham"] == "spam").mean()

np.float64(0.5092834262664611)

## Part 2: Design a Feature Extractor

Design a feature extractor for this data set and write out two files of features based on the text.
Don't forget that both the Subject and Message columns are relevant sources of text data.
For each email, you should count the number of repetitions of each feature present.
The auto-grader will assume that you are using a multinomial distribution in the following problems.

In [5]:
def extract_features(text):
    """Extract features from email text."""
    if pd.isna(text):
        text = ""
    text = str(text).lower()
    
    # Handle encoding issues - remove invalid characters
    text = text.encode('utf-8', errors='ignore').decode('utf-8')
    
    features = defaultdict(int)
    
    # Word-based features
    words = re.findall(r'\b\w+\b', text)
    for word in words:
        # Only add word if it's valid ASCII/UTF-8
        try:
            word.encode('ascii')  # Check if ASCII
            features[f'word_{word}'] += 1
        except UnicodeEncodeError:
            # Skip non-ASCII words to avoid encoding issues
            pass
    
    # Character-level features
    features['num_uppercase'] = sum(1 for c in text if c.isupper())
    features['num_digits'] = sum(1 for c in text if c.isdigit())
    features['num_punctuation'] = sum(1 for c in text if c in '!?.')
    features['num_dollar_signs'] = text.count('$')
    features['num_urls'] = len(re.findall(r'http[s]?://', text))
    
    # Common spam indicators
    spam_words = ['click', 'free', 'winner', 'congratulations', 'urgent', 'act', 'now', 
                  'offer', 'limited', 'guarantee', 'best', 'guarantee', 'no', 'risk']
    for word in spam_words:
        if word in words:
            features[f'spam_word_{word}'] += 1
    
    return dict(features)

# Test the feature extractor
test_text = enron_spam_data.iloc[0]['Message']
test_features = extract_features(test_text)
print(f"Number of features extracted: {len(test_features)}")
print(f"Sample features: {dict(list(test_features.items())[:5])}")

Number of features extracted: 5
Sample features: {'num_uppercase': 0, 'num_digits': 0, 'num_punctuation': 0, 'num_dollar_signs': 0, 'num_urls': 0}


In [6]:
# Extract features for all emails
enron_spam_data['features'] = enron_spam_data.apply(
    lambda row: extract_features(str(row['Subject']) + ' ' + str(row['Message'])), 
    axis=1
)

# Split into train and test
train_data = enron_spam_data[enron_spam_data['Message ID'] % 30 != 0].copy()
test_data = enron_spam_data[enron_spam_data['Message ID'] % 30 == 0].copy()

print(f"Training set size: {len(train_data)}")
print(f"Test set size: {len(test_data)}")

# Save train features to TSV with Unix line endings
with open('train-features.tsv', 'w', encoding='utf-8', newline='') as f:
    f.write('Message ID\tfeatures_json\n')
    for idx, row in train_data.iterrows():
        msg_id = row['Message ID']
        features_json = json.dumps(row['features'])
        f.write(f'{msg_id}\t{features_json}\n')

# Save test features to TSV with Unix line endings
with open('test-features.tsv', 'w', encoding='utf-8', newline='') as f:
    f.write('Message ID\tfeatures_json\n')
    for idx, row in test_data.iterrows():
        msg_id = row['Message ID']
        features_json = json.dumps(row['features'])
        f.write(f'{msg_id}\t{features_json}\n')

print("Feature files saved: train-features.tsv, test-features.tsv")

Training set size: 32592
Test set size: 1124
Feature files saved: train-features.tsv, test-features.tsv


Submit "train-features.tsv" and "test-features.tsv" in Gradescope.

Hint: these features will be graded based on the test accuracy of a logistic regression based on the training features.
This is to make sure that your feature set is not degenerate; you do not need to compute this regression yourself.
You can separately assess your feature quality based on your results in part 6.

In [7]:
# Compute conditional probabilities with additive smoothing
alpha = 1.0

# Collect all features and their counts for ham and spam
ham_feature_counts = defaultdict(float)
spam_feature_counts = defaultdict(float)
all_features = set()

for idx, row in train_data.iterrows():
    is_spam = row['Spam/Ham'] == 'spam'
    features = row['features']
    
    for feature, count in features.items():
        all_features.add(feature)
        if is_spam:
            spam_feature_counts[feature] += count
        else:
            ham_feature_counts[feature] += count

# Total counts for smoothing
ham_total = sum(ham_feature_counts.values())
spam_total = sum(spam_feature_counts.values())
num_features = len(all_features)

print(f"Total ham feature count: {ham_total}")
print(f"Total spam feature count: {spam_total}")
print(f"Total unique features: {num_features}")

# Compute conditional probabilities with additive smoothing
feature_probabilities = {}
for feature in all_features:
    ham_count = ham_feature_counts.get(feature, 0)
    spam_count = spam_feature_counts.get(feature, 0)
    
    ham_prob = (ham_count + alpha) / (ham_total + alpha * num_features)
    spam_prob = (spam_count + alpha) / (spam_total + alpha * num_features)
    
    feature_probabilities[feature] = {
        'ham_probability': ham_prob,
        'spam_probability': spam_prob
    }

# Save to TSV with Unix line endings and UTF-8 encoding
with open('feature-probabilities.tsv', 'w', encoding='utf-8', newline='') as f:
    f.write('feature\tham_probability\tspam_probability\n')
    for feature, probs in sorted(feature_probabilities.items()):
        f.write(f'{feature}\t{probs["ham_probability"]}\t{probs["spam_probability"]}\n')

print("Feature probabilities saved: feature-probabilities.tsv")

Total ham feature count: 5492539.0
Total spam feature count: 4282795.0
Total unique features: 154312
Feature probabilities saved: feature-probabilities.tsv


Save the conditional probabilities in a file "feature-probabilities.tsv" with columns feature, ham_probability and spam_probability.

Submit "feature-probabilities.tsv" in Gradescope.

In [8]:
# Compute prior probabilities and define naive Bayes classifier
num_ham = (train_data['Spam/Ham'] == 'ham').sum()
num_spam = (train_data['Spam/Ham'] == 'spam').sum()
prior_ham = num_ham / len(train_data)
prior_spam = num_spam / len(train_data)

print(f"Prior P(ham): {prior_ham}")
print(f"Prior P(spam): {prior_spam}")

def predict_naive_bayes(features_dict):
    """Predict spam probability using naive Bayes with numerical stability."""
    # Use log probabilities to avoid underflow
    log_prob_ham = np.log(prior_ham)
    log_prob_spam = np.log(prior_spam)
    
    for feature, count in features_dict.items():
        if feature in feature_probabilities:
            probs = feature_probabilities[feature]
            # Multinomial: feature appears 'count' times
            log_prob_ham += count * np.log(probs['ham_probability'])
            log_prob_spam += count * np.log(probs['spam_probability'])
        else:
            # Handle unknown features with smoothing
            ham_prob = alpha / (ham_total + alpha * num_features)
            spam_prob = alpha / (spam_total + alpha * num_features)
            log_prob_ham += count * np.log(ham_prob)
            log_prob_spam += count * np.log(spam_prob)
    
    # Use log-sum-exp trick for numerical stability
    # P(spam|x) = 1 / (1 + exp(log_prob_ham - log_prob_spam))
    if np.isfinite(log_prob_spam) and np.isfinite(log_prob_ham):
        log_diff = log_prob_ham - log_prob_spam
        if log_diff > 100:  # log_prob_ham >> log_prob_spam
            prob_spam = 0.0
            prob_ham = 1.0
        elif log_diff < -100:  # log_prob_spam >> log_prob_ham
            prob_spam = 1.0
            prob_ham = 0.0
        else:
            prob_spam = 1.0 / (1.0 + np.exp(log_diff))
            prob_ham = 1.0 - prob_spam
    else:
        # If either is inf, assign to that class
        if log_prob_spam > log_prob_ham:
            prob_spam = 1.0
            prob_ham = 0.0
        else:
            prob_spam = 0.0
            prob_ham = 1.0
    
    return prob_ham, prob_spam

# Test the classifier
test_prob_ham, test_prob_spam = predict_naive_bayes(train_data.iloc[0]['features'])
print(f"Test prediction - Ham: {test_prob_ham:.4f}, Spam: {test_prob_spam:.4f}")

Prior P(ham): 0.49070324005891014
Prior P(spam): 0.5092967599410898
Test prediction - Ham: 1.0000, Spam: 0.0000


In [9]:
# Make predictions on test data
test_predictions = []
for idx, row in test_data.iterrows():
    msg_id = row['Message ID']
    prob_ham, prob_spam = predict_naive_bayes(row['features'])
    test_predictions.append({
        'Message ID': msg_id,
        'ham': prob_ham,
        'spam': prob_spam,
        'true_label': row['Spam/Ham']
    })

# Save to TSV with Unix line endings
with open('test-predictions.tsv', 'w', encoding='utf-8', newline='') as f:
    f.write('Message ID\tham\tspam\n')
    for pred in test_predictions:
        f.write(f'{pred["Message ID"]}\t{pred["ham"]}\t{pred["spam"]}\n')

print(f"Test predictions saved: {len(test_predictions)} emails")
print("Sample predictions:")
for pred in test_predictions[:3]:
    print(f"  Message {pred['Message ID']} (true: {pred['true_label']}): ham={pred['ham']:.4f}, spam={pred['spam']:.4f}")

Test predictions saved: 1124 emails
Sample predictions:
  Message 0 (true: ham): ham=0.0037, spam=0.9963
  Message 30 (true: ham): ham=1.0000, spam=0.0000
  Message 60 (true: ham): ham=1.0000, spam=0.0000


In [10]:
# Make predictions on training data
train_predictions = []
for idx, row in train_data.iterrows():
    msg_id = row['Message ID']
    prob_ham, prob_spam = predict_naive_bayes(row['features'])
    train_predictions.append({
        'Message ID': msg_id,
        'ham': prob_ham,
        'spam': prob_spam
    })

# Save to TSV with Unix line endings
with open('train-predictions.tsv', 'w', encoding='utf-8', newline='') as f:
    f.write('Message ID\tham\tspam\n')
    for pred in train_predictions:
        f.write(f'{pred["Message ID"]}\t{pred["ham"]}\t{pred["spam"]}\n')

print(f"Training predictions saved: {len(train_predictions)} emails")
print("Sample predictions:")
for pred in train_predictions[:5]:
    print(f"  Message {pred['Message ID']}: ham={pred['ham']:.4f}, spam={pred['spam']:.4f}")

Training predictions saved: 32592 emails
Sample predictions:
  Message 1: ham=1.0000, spam=0.0000
  Message 2: ham=1.0000, spam=0.0000
  Message 3: ham=1.0000, spam=0.0000
  Message 4: ham=1.0000, spam=0.0000
  Message 5: ham=1.0000, spam=0.0000


Save this data in a file "roc.tsv" with columns threshold, false_positive_rate and true_positive rate.

In [11]:
# Validation and summary
import os

print("=" * 60)
print("PROJECT SUMMARY")
print("=" * 60)

# Check all output files exist
output_files = [
    'train-features.tsv',
    'test-features.tsv',
    'feature-probabilities.tsv',
    'train-predictions.tsv',
    'test-predictions.tsv',
    'roc.tsv'
]

for filename in output_files:
    if os.path.exists(filename):
        size = os.path.getsize(filename)
        print(f"✓ {filename} ({size} bytes)")
    else:
        print(f"✗ {filename} NOT FOUND")

print("\n" + "=" * 60)
print("DATA STATISTICS")
print("=" * 60)
print(f"Training set: {len(train_data)} emails ({(train_data['Spam/Ham']=='spam').sum()} spam)")
print(f"Test set: {len(test_data)} emails ({(test_data['Spam/Ham']=='spam').sum()} spam)")
print(f"Total features: {len(all_features)}")
print(f"Feature extraction method: word counts + character features + spam indicators")
print(f"Smoothing: Additive smoothing with alpha=1.0")

PROJECT SUMMARY
✓ train-features.tsv (74035640 bytes)
✓ test-features.tsv (2429344 bytes)
✓ feature-probabilities.tsv (9100124 bytes)
✓ train-predictions.tsv (758242 bytes)
✓ test-predictions.tsv (26656 bytes)
✗ roc.tsv NOT FOUND

DATA STATISTICS
Training set: 32592 emails (16599 spam)
Test set: 1124 emails (572 spam)
Total features: 154312
Feature extraction method: word counts + character features + spam indicators
Smoothing: Additive smoothing with alpha=1.0


In [12]:
# Compute ROC curve for test data
thresholds = np.arange(0.01, 1.00, 0.01)
roc_data = []

for threshold in thresholds:
    # Count true positives, false positives, etc.
    true_positives = 0
    false_positives = 0
    true_negatives = 0
    false_negatives = 0
    
    for pred in test_predictions:
        predicted_spam = pred['spam'] >= threshold
        is_spam = pred['true_label'] == 'spam'
        
        if predicted_spam and is_spam:
            true_positives += 1
        elif predicted_spam and not is_spam:
            false_positives += 1
        elif not predicted_spam and not is_spam:
            true_negatives += 1
        else:  # not predicted_spam and is_spam
            false_negatives += 1
    
    # Calculate rates
    total_negatives = false_positives + true_negatives
    total_positives = true_positives + false_negatives
    
    if total_negatives > 0:
        false_positive_rate = false_positives / total_negatives
    else:
        false_positive_rate = 0
    
    if total_positives > 0:
        true_positive_rate = true_positives / total_positives
    else:
        true_positive_rate = 0
    
    roc_data.append({
        'threshold': threshold,
        'false_positive_rate': false_positive_rate,
        'true_positive_rate': true_positive_rate
    })

# Save ROC curve to TSV with Unix line endings
with open('roc.tsv', 'w', encoding='utf-8', newline='') as f:
    f.write('threshold\tfalse_positive_rate\ttrue_positive_rate\n')
    for row in roc_data:
        f.write(f'{row["threshold"]:.2f}\t{row["false_positive_rate"]}\t{row["true_positive_rate"]}\n')

print("ROC curve saved: roc.tsv")
print("Sample ROC points:")
for row in roc_data[::10]:  # Print every 10th point
    print(f"  Threshold {row['threshold']:.2f}: FPR={row['false_positive_rate']:.4f}, TPR={row['true_positive_rate']:.4f}")

ROC curve saved: roc.tsv
Sample ROC points:
  Threshold 0.01: FPR=0.0362, TPR=0.9983
  Threshold 0.11: FPR=0.0272, TPR=0.9983
  Threshold 0.21: FPR=0.0272, TPR=0.9983
  Threshold 0.31: FPR=0.0236, TPR=0.9948
  Threshold 0.41: FPR=0.0236, TPR=0.9948
  Threshold 0.51: FPR=0.0236, TPR=0.9948
  Threshold 0.61: FPR=0.0217, TPR=0.9948
  Threshold 0.71: FPR=0.0199, TPR=0.9948
  Threshold 0.81: FPR=0.0199, TPR=0.9913
  Threshold 0.91: FPR=0.0199, TPR=0.9860


## Part 7: Signup for Gemini API Key

Create a free Gemini API key at https://aistudio.google.com/app/api-keys.
You will need to do this with a personal Google account - it will not work with your BU Google account.
This will not incur any charges unless you configure billing information for the key.

You will be asked to start a Gemini free trial for week 11.
This will not incur any charges unless you exceed expected usage by an order of magnitude.


No submission needed.

## Part 8: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 9: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.